In [ ]:
!pip install -q fastapi uvicorn pydantic nest-asyncio pyngrok
!apt-get update -qq && apt-get install -qq ffmpeg
"""
Colab FastAPI 视频生成脚本 (纯净 Drive 同步版)。
挂载云盘、暴露API并写入地址、接受生成命令、直接在云盘任务目录下执行FFmpeg。
"""

import os
import shlex
import threading
import subprocess
from typing import Dict, Optional, Tuple

import nest_asyncio
import uvicorn
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
from pyngrok import ngrok

# ==========================================
# 配置区域 (Configuration)
# ==========================================
SERVER_PORT = 8000
NGROK_AUTH_TOKEN = "39x8NCc9aFTPtQ9eCWAjfFuN756_7CktGrfGGEDcUpoTawjSh"

# Google Drive 挂载与联动配置
GDRIVE_MOUNT_POINT = "/content/drive"
DRIVE_BASE_DIR = "/content/drive/MyDrive/AiGenerations"
API_PATH_FILE = os.path.join(DRIVE_BASE_DIR, "API_PATH.txt")

# 全局任务数据库
TASK_DB: Dict[str, Dict[str, str]] = {}

app = FastAPI(title="Video Generation API")
nest_asyncio.apply()

# ==========================================
# 数据模型定义 (Data Models)
# ==========================================
class StartTaskRequest(BaseModel):
    task_id: str
    ffmpeg_command: str

class StartTaskResponse(BaseModel):
    task_id: str

class QueryTaskResponse(BaseModel):
    task_id: str
    status: str
    error: Optional[str] = None
    result_url: Optional[str] = None

# ==========================================
# 底层功能函数 (Core Functions)
# ==========================================
def print_log(msg: str) -> None:
    """带强制刷新的日志打印。"""
    print(msg, flush=True)

def mount_google_drive_and_init() -> bool:
    print_log("[Environment] 准备挂载 Google Drive...")
    try:
        from google.colab import drive
        drive.mount(GDRIVE_MOUNT_POINT)
        print_log(f"[Environment] [成功] Google Drive 挂载至: {GDRIVE_MOUNT_POINT}")
        if not os.path.exists(DRIVE_BASE_DIR):
            os.makedirs(DRIVE_BASE_DIR)
        return True
    except Exception as exc:
        print_log(f"[Environment] [失败] Google Drive 挂载异常: {exc}")
        return False

def start_ngrok_tunnel() -> str:
    print_log(f"[Network] 准备启动 ngrok 隧道，端口: {SERVER_PORT}")
    try:
        if NGROK_AUTH_TOKEN: ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        public_url = ngrok.connect(SERVER_PORT).public_url
        print_log(f"[Network] [成功] API 公网地址: {public_url}")
        return public_url
    except Exception as exc:
        print_log(f"[Network] [失败] 启动 ngrok 隧道失败: {exc}")
        return ""

def write_api_url_to_drive(url: str) -> bool:
    print_log(f"[File] 准备写入 API 地址至: {API_PATH_FILE}")
    try:
        with open(API_PATH_FILE, 'w', encoding='utf-8') as f: f.write(url)
        return True
    except Exception as exc:
        print_log(f"[File] [失败] 写入 API 地址异常: {exc}")
        return False

def run_fastapi_server() -> None:
    print_log("[Server] Uvicorn 服务器开始运行...")
    try:
        uvicorn.run(app, host="0.0.0.0", port=SERVER_PORT, log_level="info")
    except Exception as exc:
        print_log(f"[Server] [失败] Uvicorn 运行异常: {exc}")

def start_server_in_thread() -> bool:
    try:
        server_thread = threading.Thread(target=run_fastapi_server, daemon=True)
        server_thread.start()
        return True
    except Exception as exc:
        print_log(f"[Server] [失败] 线程启动异常: {exc}")
        return False

def update_task_state(task_id: str, status: str, error: str = "", result_url: str = "") -> bool:
    print_log(f"[State] 更新任务 {task_id} 状态为: {status}")
    if task_id not in TASK_DB: return False
    TASK_DB[task_id]["status"] = status
    if error: TASK_DB[task_id]["error"] = error
    if result_url: TASK_DB[task_id]["result_url"] = result_url
    return True

def find_source_image(task_dir: str, task_id: str) -> Optional[str]:
    print_log(f"[File] 在目录 {task_dir} 寻找源图片...")
    try:
        if not os.path.exists(task_dir):
            print_log(f"[File] [警告] 任务目录尚未同步: {task_dir}")
            if os.path.exists(DRIVE_BASE_DIR):
                print_log(f"[File] [诊断] 根目录内容: {os.listdir(DRIVE_BASE_DIR)[:10]}")
            return None
        files = os.listdir(task_dir)
        for f in files:
            if f.startswith(task_id) and f.lower().endswith(('.png', '.jpg', '.jpeg')):
                return os.path.join(task_dir, f)
        print_log(f"[File] [失败] 未找到匹配图片。当前目录文件: {files}")
        return None
    except Exception as exc:
        print_log(f"[File] [失败] 查找异常: {exc}")
        return None

def execute_ffmpeg(command_str: str, log_file: str) -> Tuple[bool, str]:
    """执行 FFmpeg 并返回 (是否成功, 错误详情/结果)。同时写入日志文件。"""
    print_log(f"[FFmpeg] 准备执行: {command_str}")
    try:
        cmd_list = shlex.split(command_str)
        process = subprocess.run(cmd_list, capture_output=True, text=True, check=False)
        
        full_log = f"Command: {command_str}\nReturn Code: {process.returncode}\nStdout: {process.stdout}\nStderr: {process.stderr}"
        try:
            with open(log_file, 'w', encoding='utf-8') as f: f.write(full_log)
            print_log(f"[FFmpeg] [日志] 已写入本地: {log_file}")
        except Exception as log_exc:
            print_log(f"[FFmpeg] [警告] 无法写入日志文件: {log_exc}")

        if process.returncode != 0:
            err_msg = process.stderr.strip() if process.stderr else f"Process exited with code {process.returncode}"
            print_log(f"[FFmpeg] [失败] 状态码: {process.returncode}\n{err_msg}")
            return False, err_msg
            
        print_log("[FFmpeg] [成功] 任务执行完毕。")
        return True, ""
    except Exception as exc:
        err_str = f"执行崩溃: {exc}"
        print_log(f"[FFmpeg] [失败] {err_str}")
        return False, err_str

# ==========================================
# 任务处理逻辑 (Task Processing Pipeline)
# ==========================================
def process_video_generation(task_id: str, ffmpeg_template: str) -> None:
    print_log(f"\n---> [Pipeline] 任务 {task_id} 启动 <---")
    task_dir = os.path.join(DRIVE_BASE_DIR, task_id)
    log_file = os.path.join(task_dir, "task_log.txt")
    
    import time
    image_path = None
    for attempt in range(12):
        image_path = find_source_image(task_dir, task_id)
        if image_path: break
        print_log(f"[Pipeline] [等待] 云端正在同步... ({attempt+1}/12)")
        time.sleep(5)
        
    if not image_path:
        update_task_state(task_id, "fail", error="未在云盘找到图片文件，请检查本地 G 盘是否同步成功。")
        return
        
    video_path = os.path.join(task_dir, f"{task_id}.mp4")
    final_command = ffmpeg_template.replace("{INPUT_IMAGE}", image_path).replace("{OUTPUT_VIDEO}", video_path)
    
    is_success, error_detail = execute_ffmpeg(final_command, log_file)
    if not is_success:
        update_task_state(task_id, "fail", error=f"FFmpeg 执行失败: {error_detail}")
        return
        
    print_log(f"[Pipeline] [成功] 视频已保存至: {video_path}")
    update_task_state(task_id, "success", result_url=video_path)

# ==========================================
# FastAPI 路由定义 (API Routes)
# ==========================================
@app.post("/start", response_model=StartTaskResponse)
def api_start(request: StartTaskRequest, background_tasks: BackgroundTasks):
    task_id = request.task_id
    print_log(f"\n[API] 接收到 /start 请求，任务: {task_id}")
    TASK_DB[task_id] = {"status": "running", "error": "", "result_url": ""}
    background_tasks.add_task(process_video_generation, task_id, request.ffmpeg_command)
    return StartTaskResponse(task_id=task_id)

@app.get("/query", response_model=QueryTaskResponse)
def api_query(task_id: str):
    if task_id not in TASK_DB: return QueryTaskResponse(task_id=task_id, status="fail", error="Not found")
    t = TASK_DB[task_id]
    return QueryTaskResponse(task_id=task_id, status=t["status"], error=t.get("error"), result_url=t.get("result_url"))

def main() -> None:
    if not mount_google_drive_and_init(): return
    public_url = start_ngrok_tunnel()
    if not public_url: return
    if not write_api_url_to_drive(public_url): return
    if not start_server_in_thread(): return
    print_log("\n[Main] 服务就绪，等待本地脚本轮询...\n")

if __name__ == "__main__":
    main()